In [41]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

In [42]:
columns_to_keep = [
    'region', 'parent_category_name', 'category_name',
    'param_1', 'param_2', 'param_3', 'price', 'item_seq_number',
    'user_type', 'description', 'image', 'deal_probability'
]

dtypes = {
    'region': 'category',
    'parent_category_name': 'category',
    'category_name': 'category',
    'user_type': 'category',
    'price': 'float32',
    'item_seq_number': 'int32',
    'deal_probability': 'float32'
}

df=pd.read_csv('../train.csv', usecols=columns_to_keep, dtype=dtypes, nrows=50000)

df.head()

,region,parent_category_name,category_name,param_1,param_2,param_3,description,price,item_seq_number,user_type,image,deal_probability
0,Свердловская область,Личные вещи,Товары для детей и игрушки,Постельные принадлежности,NaN,NaN,"Кокон для сна малыша,пользовались меньше месяц...",400.0,2,Private,d10c7e016e03247a3bf2d13348fe959fe6f436c1caf64c...,0.12789
1,Самарская область,Для дома и дачи,Мебель и интерьер,Другое,NaN,NaN,"Стойка для одежды, под вешалки. С бутика.",3000.0,19,Private,79c9392cc51a9c81c6eb91eceb8e552171db39d7142700...,0.00000
2,Ростовская область,Бытовая электроника,Аудио и видео,"Видео, DVD и Blu-ray плееры",NaN,NaN,"В хорошем состоянии, домашний кинотеатр с blu ...",4000.0,9,Private,b7f250ee3f39e1fedd77c141f273703f4a9be59db4b48a...,0.43177
3,Татарстан,Личные вещи,Товары для детей и игрушки,Автомобильные кресла,NaN,NaN,Продам кресло от0-25кг,2200.0,286,Company,e6ef97e0725637ea84e3d203e82dadb43ed3cc0a1c8413...,0.80323
4,Волгоградская область,Транспорт,Автомобили,С пробегом,ВАЗ (LADA),2110,Все вопросы по телефону.,40000.0,3,Private,54a687a3a0fc1d68aed99bdaaf551c5c70b761b16fd0a2...,0.20797


In [43]:
df['has_params']=df[['param_1', 'param_2', 'param_3']].notna().any(axis=1).astype('int8')

df['param_1_exists']=df['param_1'].notna().astype('int8')
df['param_2_exists']=df['param_2'].notna().astype('int8')
df['param_3_exists']=df['param_3'].notna().astype('int8')

df['param_1']=df['param_1'].fillna("")
df['param_2']=df['param_2'].fillna("")
df['param_3']=df['param_3'].fillna("")

df['image']=df['image'].notna().astype('int8')

df['description']=df['description'].fillna("")
df['description_len']=df['description'].str.len().astype('int32')

bins=[-1, 10, 50, 250, 1000, np.inf]
labels=['1. Пусто', '2. Очень короткое', '3. Оптимальное', '4. Подробное', '5. Слишком длинное']

df['description_group']=pd.cut(df['description_len'], bins=bins, labels=labels)
df.head()

,region,parent_category_name,category_name,param_1,param_2,param_3,description,price,item_seq_number,user_type,image,deal_probability,has_params,param_1_exists,param_2_exists,param_3_exists,description_len,description_group
0,Свердловская область,Личные вещи,Товары для детей и игрушки,Постельные принадлежности,,,"Кокон для сна малыша,пользовались меньше месяц...",400.0,2,Private,1,0.12789,1,1,0,0,58,3. Оптимальное
1,Самарская область,Для дома и дачи,Мебель и интерьер,Другое,,,"Стойка для одежды, под вешалки. С бутика.",3000.0,19,Private,1,0.00000,1,1,0,0,41,2. Очень короткое
2,Ростовская область,Бытовая электроника,Аудио и видео,"Видео, DVD и Blu-ray плееры",,,"В хорошем состоянии, домашний кинотеатр с blu ...",4000.0,9,Private,1,0.43177,1,1,0,0,99,3. Оптимальное
3,Татарстан,Личные вещи,Товары для детей и игрушки,Автомобильные кресла,,,Продам кресло от0-25кг,2200.0,286,Company,1,0.80323,1,1,0,0,22,2. Очень короткое
4,Волгоградская область,Транспорт,Автомобили,С пробегом,ВАЗ (LADA),2110,Все вопросы по телефону.,40000.0,3,Private,1,0.20797,1,1,1,1,24,2. Очень короткое


In [44]:
df.isnull().sum()

region                     0
parent_category_name       0
category_name              0
param_1                    0
param_2                    0
param_3                    0
description                0
price                   2918
item_seq_number            0
user_type                  0
image                      0
deal_probability           0
has_params                 0
param_1_exists             0
param_2_exists             0
param_3_exists             0
description_len            0
description_group          0
dtype: int64

In [45]:
print("Самые дешевые товары:")
print(df[df['price'] > 0][['category_name', 'price']].sort_values(by='price').head(10))

print("\nСамые дорогие товары:")
print(df[['category_name', 'price']].sort_values(by='price', ascending=False).head(10))

Самые дешевые товары:
                 category_name  price
29105       Красота и здоровье    1.0
6592        Коллекционирование    1.0
45721        Предложение услуг    1.0
19301                 Телефоны    1.0
29399  Оргтехника и расходники    1.0
190     Ремонт и строительство    1.0
28077   Ремонт и строительство    1.0
49282        Предложение услуг    1.0
6359         Предложение услуг    1.0
13966     Дома, дачи, коттеджи    1.0

Самые дорогие товары:
                   category_name         price
47509                   Квартиры  1.170000e+09
14369                   Телефоны  4.001003e+08
43325  Коммерческая недвижимость  3.230000e+08
9244        Дома, дачи, коттеджи  2.500000e+08
5805        Дома, дачи, коттеджи  1.500000e+08
27508  Коммерческая недвижимость  1.200000e+08
35670  Коммерческая недвижимость  1.086045e+08
12668          Предложение услуг  1.000000e+08
16287  Коммерческая недвижимость  9.500000e+07
22593  Коммерческая недвижимость  8.368402e+07


In [46]:
df[['category_name', 'price']]

,category_name,price
0,Товары для детей и игрушки,400.0
1,Мебель и интерьер,3000.0
2,Аудио и видео,4000.0
3,Товары для детей и игрушки,2200.0
4,Автомобили,40000.0
...,...,...
49995,Квартиры,3500000.0
49996,Другие животные,500.0
49997,"Одежда, обувь, аксессуары",500.0
49998,Квартиры,3000.0


In [47]:
df.head()

,region,parent_category_name,category_name,param_1,param_2,param_3,description,price,item_seq_number,user_type,image,deal_probability,has_params,param_1_exists,param_2_exists,param_3_exists,description_len,description_group
0,Свердловская область,Личные вещи,Товары для детей и игрушки,Постельные принадлежности,,,"Кокон для сна малыша,пользовались меньше месяц...",400.0,2,Private,1,0.12789,1,1,0,0,58,3. Оптимальное
1,Самарская область,Для дома и дачи,Мебель и интерьер,Другое,,,"Стойка для одежды, под вешалки. С бутика.",3000.0,19,Private,1,0.00000,1,1,0,0,41,2. Очень короткое
2,Ростовская область,Бытовая электроника,Аудио и видео,"Видео, DVD и Blu-ray плееры",,,"В хорошем состоянии, домашний кинотеатр с blu ...",4000.0,9,Private,1,0.43177,1,1,0,0,99,3. Оптимальное
3,Татарстан,Личные вещи,Товары для детей и игрушки,Автомобильные кресла,,,Продам кресло от0-25кг,2200.0,286,Company,1,0.80323,1,1,0,0,22,2. Очень короткое
4,Волгоградская область,Транспорт,Автомобили,С пробегом,ВАЗ (LADA),2110,Все вопросы по телефону.,40000.0,3,Private,1,0.20797,1,1,1,1,24,2. Очень короткое


In [48]:
q_low = df.groupby('category_name')['price'].transform('quantile', 0.01)
q_high = df.groupby('category_name')['price'].transform('quantile', 0.99)

df_cleaned= df[
    ((df['price'] >= q_low) & (df['price'] <= q_high)) | 
    df['price'].isnull()
]

print(f"Было строк: {len(df)}, осталось: {len(df_cleaned)}")

Было строк: 50000, осталось: 49167


In [49]:
df_cleaned.head()

,region,parent_category_name,category_name,param_1,param_2,param_3,description,price,item_seq_number,user_type,image,deal_probability,has_params,param_1_exists,param_2_exists,param_3_exists,description_len,description_group
0,Свердловская область,Личные вещи,Товары для детей и игрушки,Постельные принадлежности,,,"Кокон для сна малыша,пользовались меньше месяц...",400.0,2,Private,1,0.12789,1,1,0,0,58,3. Оптимальное
1,Самарская область,Для дома и дачи,Мебель и интерьер,Другое,,,"Стойка для одежды, под вешалки. С бутика.",3000.0,19,Private,1,0.00000,1,1,0,0,41,2. Очень короткое
2,Ростовская область,Бытовая электроника,Аудио и видео,"Видео, DVD и Blu-ray плееры",,,"В хорошем состоянии, домашний кинотеатр с blu ...",4000.0,9,Private,1,0.43177,1,1,0,0,99,3. Оптимальное
3,Татарстан,Личные вещи,Товары для детей и игрушки,Автомобильные кресла,,,Продам кресло от0-25кг,2200.0,286,Company,1,0.80323,1,1,0,0,22,2. Очень короткое
4,Волгоградская область,Транспорт,Автомобили,С пробегом,ВАЗ (LADA),2110,Все вопросы по телефону.,40000.0,3,Private,1,0.20797,1,1,1,1,24,2. Очень короткое


In [50]:
columns_to_drop=['description']

df_cleaned=df_cleaned.drop(columns=columns_to_drop)

df_cleaned.to_csv('train_prepared.csv', index=False)
